In [13]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from typing_extensions import NotRequired
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
import operator

model = ChatOllama(model='llama3.2:latest')

In [3]:
class EvaluationSchema(BaseModel):
    
    feedback: str = Field (description= 'Detailed feedback for essay')
    score: int = Field(description= 'Score out of 10', ge=0, le=10)

In [4]:
structured_model = model.with_structured_output(EvaluationSchema)

In [5]:
essay = """# **The Rise of Artificial Intelligence in India**

## **Outline**

1. Introduction
2. Understanding Artificial Intelligence
3. Growth of AI in India
4. Government Initiatives Supporting AI
5. AI in Different Sectors

   * Healthcare
   * Education
   * Agriculture
   * Finance
   * Manufacturing
6. Benefits of AI in India
7. Challenges and Concerns
8. Future of AI in India
9. Conclusion

---

# **The Rise of Artificial Intelligence in India**

## **Introduction**

Artificial Intelligence (AI) has become one of the most transformative technologies of the 21st century. It enables machines to perform tasks that usually require human intelligence, such as learning, reasoning, problem-solving, and decision-making. Over the past decade, India has witnessed remarkable growth in AI adoption across industries, businesses, and public services. With its large pool of skilled professionals, expanding digital infrastructure, and growing startup ecosystem, India is emerging as a global hub for AI innovation. The rise of AI is reshaping the economy, improving public services, and creating new opportunities for businesses and individuals alike.

## **Understanding Artificial Intelligence**

Artificial Intelligence refers to the simulation of human intelligence by machines. AI systems use technologies such as machine learning, deep learning, natural language processing, and computer vision to analyze data, recognize patterns, and make informed decisions. These systems become more effective over time as they learn from experience.

Today, AI powers many everyday applications, including voice assistants, recommendation systems on streaming platforms, online shopping suggestions, chatbots, navigation apps, and facial recognition technology. As AI continues to evolve, it is becoming an essential part of modern life.

## **Growth of AI in India**

India's AI journey has accelerated rapidly due to increasing internet penetration, affordable smartphones, cloud computing, and the availability of vast amounts of digital data. Indian technology companies, startups, educational institutions, and research organizations are investing heavily in AI research and development.

The country has become home to thousands of AI startups working on innovative solutions in healthcare, agriculture, education, cybersecurity, logistics, and financial technology. Global technology companies have also established AI research centers in India, recognizing the country's engineering talent and growing digital economy.

Furthermore, India's strong IT industry has played a significant role in integrating AI into business operations, helping organizations automate routine tasks and improve productivity.

## **Government Initiatives Supporting AI**

The Government of India has recognized AI as a strategic technology for national development. Several initiatives have been introduced to encourage AI research, innovation, and responsible adoption. Policies promoting digital transformation, startup incubation, skill development, and AI-based public services have strengthened the country's AI ecosystem.

Government programs aim to use AI for improving governance, healthcare delivery, education, agriculture, smart cities, transportation, and disaster management. Educational institutions are also introducing AI-related courses to prepare students for future careers.

Collaboration between government agencies, academic institutions, private companies, and startups is helping create an environment where AI innovation can flourish.

## **AI in Different Sectors**

### Healthcare

Artificial Intelligence is revolutionizing healthcare in India. AI-powered systems assist doctors in diagnosing diseases, interpreting medical images, predicting health risks, and recommending treatments. Hospitals use AI to manage patient records, schedule appointments, and optimize resource allocation. Telemedicine platforms equipped with AI are making healthcare more accessible in rural and remote areas.

### Education

In education, AI enables personalized learning experiences by adapting lessons according to each student's strengths and weaknesses. Intelligent tutoring systems provide instant feedback, while AI-powered language translation tools help students learn in multiple languages. Teachers also benefit from AI by automating administrative tasks and assessing student performance more efficiently.

### Agriculture

Agriculture remains one of India's most important sectors, and AI is helping farmers improve productivity. AI analyzes weather conditions, soil quality, crop health, and irrigation needs. Farmers receive timely recommendations on fertilizer usage, pest control, and harvesting schedules. These innovations increase crop yields while reducing costs and minimizing environmental impact.

### Finance

The financial sector has widely adopted AI for fraud detection, customer service, credit assessment, and investment management. Banks use AI-powered chatbots to answer customer queries around the clock. Machine learning algorithms analyze transaction patterns to detect suspicious activities and reduce financial fraud.

### Manufacturing

AI is transforming manufacturing by automating production processes, predicting equipment failures, improving quality control, and optimizing supply chains. Smart factories equipped with AI technologies can operate more efficiently, reduce waste, and enhance workplace safety.

## **Benefits of AI in India**

The rise of AI offers numerous benefits for India's economic and social development. One major advantage is increased productivity. AI automates repetitive tasks, allowing employees to focus on creative and strategic work. Businesses benefit from improved efficiency, reduced operational costs, and faster decision-making.

AI also supports better public services. Governments can use AI to improve traffic management, monitor environmental conditions, enhance public safety, and deliver welfare programs more effectively.

Another significant benefit is job creation. Although AI automates certain tasks, it also creates new career opportunities in AI development, data science, cybersecurity, robotics, cloud computing, and digital transformation. As demand for AI professionals grows, educational institutions are expanding training programs to equip students with relevant skills.

Additionally, AI encourages innovation and entrepreneurship. Indian startups are developing AI-based solutions that address local challenges while competing in international markets.

## **Challenges and Concerns**

Despite its many advantages, AI presents several challenges. One concern is job displacement, as automation may reduce the demand for certain routine occupations. Workers need continuous upskilling and reskilling to remain competitive in the changing job market.

Data privacy and cybersecurity are also major concerns. AI systems require large amounts of data, making it essential to protect personal information from misuse and cyberattacks.

Another challenge is algorithmic bias. AI systems can produce unfair or inaccurate outcomes if they are trained on biased data. Developers must ensure transparency, fairness, and accountability when designing AI applications.

India also faces challenges related to digital infrastructure, research funding, and access to advanced computing resources, especially in rural regions. Addressing these issues will require coordinated efforts from government, industry, and academia.

## **Future of AI in India**

The future of AI in India is highly promising. As digital adoption continues to grow, AI will play an increasingly important role in economic development and public service delivery. Emerging technologies such as generative AI, robotics, autonomous systems, and intelligent automation are expected to transform industries even further.

India has the potential to become a global leader in ethical and inclusive AI by investing in education, research, innovation, and responsible governance. Partnerships between universities, startups, multinational companies, and government organizations will continue to drive AI advancement.

Preparing the workforce through AI education and digital literacy will ensure that citizens can benefit from technological progress while minimizing its risks.

## **Conclusion**

Artificial Intelligence is transforming India by driving innovation, improving productivity, and enhancing the quality of life. From healthcare and education to agriculture and finance, AI is creating new possibilities across every sector. While challenges such as privacy, ethics, and workforce adaptation must be addressed, the overall impact of AI is overwhelmingly positive. With continued investment in research, education, infrastructure, and responsible governance, India is well-positioned to become one of the world's leading AI-powered economies. The rise of AI is not merely a technological revolution; it is a catalyst for sustainable growth, inclusive development, and a brighter future for the nation.
"""

In [7]:

prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign score out of 10 \n {essay} '
structured_model.invoke(prompt).score

7

In [31]:
#define State

class EssayAnalyzerState(TypedDict):
    
    essay: str
    coT_fb: str
    doA_fb: str    
    language_fb: str
    individual_score: Annotated[list[int], operator.add]
    
    final_fb: str
    final_score: float
    

In [32]:
def clarity_fb(state: EssayAnalyzerState):

    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign score out of 10 \n {state['essay']} '
    output = structured_model.invoke(prompt)
    
    return {'coT_fb': output.feedback, 'individual_score': [output.score]}


In [33]:
def analysis_fb(state: EssayAnalyzerState):

    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign score out of 10 \n {state['essay']} '
    output = structured_model.invoke(prompt)
    
    return {'doA_fb': output.feedback, 'individual_score': [output.score]}

In [34]:
def lang_fb(state: EssayAnalyzerState):

    prompt = f'Evaluate the quality of language of the following essay and provide a feedback and assign score out of 10 \n {state['essay']} '
    output = structured_model.invoke(prompt)
    
    return {'language_fb': output.feedback, 'individual_score': [output.score]}

In [35]:
def final_feedback(state: EssayAnalyzerState):

    prompt = f'Based on the following feedbacks create a summarized feedback \n clarity of thought feedback - {state["coT_fb"]} \n depth of analysis feedback - {state["doA_fb"]} \n language feedback - {state["language_fb"]}'
    overall_feedback = model.invoke(prompt).content
    
    avg_score = sum(state["individual_score"])/len(state["individual_score"])
    
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [39]:
# define graph

graph = StateGraph(EssayAnalyzerState)



In [40]:
#add nodes and edges

graph.add_node('clarity_fb', clarity_fb)
graph.add_node('analysis_fb', analysis_fb)
graph.add_node('lang_fb', lang_fb)
graph.add_node('final_feedback', final_feedback)

graph.add_edge(START, 'clarity_fb')
graph.add_edge(START, 'analysis_fb')
graph.add_edge(START, 'lang_fb')

graph.add_edge('clarity_fb', 'final_feedback')
graph.add_edge('analysis_fb', 'final_feedback')
graph.add_edge('lang_fb', 'final_feedback')

graph.add_edge('final_feedback', END)

In [42]:
workflow = graph.compile()

In [43]:
initial_state = {
    'essay' : essay
}

workflow.invoke(initial_state)



{'essay': "# **The Rise of Artificial Intelligence in India**\n\n## **Outline**\n\n1. Introduction\n2. Understanding Artificial Intelligence\n3. Growth of AI in India\n4. Government Initiatives Supporting AI\n5. AI in Different Sectors\n\n   * Healthcare\n   * Education\n   * Agriculture\n   * Finance\n   * Manufacturing\n6. Benefits of AI in India\n7. Challenges and Concerns\n8. Future of AI in India\n9. Conclusion\n\n---\n\n# **The Rise of Artificial Intelligence in India**\n\n## **Introduction**\n\nArtificial Intelligence (AI) has become one of the most transformative technologies of the 21st century. It enables machines to perform tasks that usually require human intelligence, such as learning, reasoning, problem-solving, and decision-making. Over the past decade, India has witnessed remarkable growth in AI adoption across industries, businesses, and public services. With its large pool of skilled professionals, expanding digital infrastructure, and growing startup ecosystem, India